# Survivor Strategy Analysis
### Identifying the 4 key variables that best predict winners

**Pipeline:**
1. Data Loading & Cleaning  
2. Variable Selection — testing all candidates, eliminating redundant ones  
3. Final 4 Variables — construction & validation  
4. Era Segmentation  
5. Logistic Regression — univariate → combined → per era  
6. Visualizations

---
## 1. Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
from scipy.stats import pearsonr
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

BLUE  = '#1565C0'
LBLUE = '#90CAF9'
RED   = '#EF5350'
DBLUE = '#0D47A1'
DRED  = '#B71C1C'
GREY  = '#78909C'

df = pd.read_excel(r"C:\Users\sruja\survivor_project\archive\Voting Stats Plus.xlsx")
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(f"Raw shape: {df.shape}")

In [ ]:
# ── Drop tribalsattended == 0 (medivac / pre-tribal boot — ratios undefined) ──
pre = len(df)
df_work = df[df['tribalsattended'] > 0].copy()
print(f"Dropped {pre - len(df_work)} players with tribalsattended=0 (medivac/pre-tribal boot).")

# ── Fix data entry error ──────────────────────────────────────────────────────
# Anna Khait (Kaoh Rong): correctlyvoted=9 with votescast=1 — impossible. Fixed to 0.
df_work.loc[df_work['playername'] == 'Anna Khait', 'correctlyvoted'] = 0
print("Fixed: Anna Khait correctlyvoted 9 -> 0 (votescast=1, entry error).")
print(f"Working rows: {len(df_work)}")

---
## 2. Variable Selection — Testing All Candidates

Before settling on 4 variables, we test every reasonable metric the data supports and eliminate candidates that are **redundant** (r > 0.90 with a better variable) or **too weak** (AUC < 0.65).

### Candidates tested
| Variable | Formula | Reasoning |
|---|---|---|
| `target_rate` | votesrecieved / tribalsattended | How often others targeted you |
| `correct_vote_ratio` | correctlyvoted / votescast | Strategic alignment with the majority |
| `immunity_rate` | individualimmunites / tribalsattended | Challenge dominance |
| `tribal_depth` | tribalsattended / (playersonseason − 1) | How deep into the season you survived |
| `pressure_index` | (votesrecieved − votesnegated) / tribalsattended | Net unprotected exposure |
| `threat_avoidance` | 1 / (1 + target_rate) | Inverse framing of being targeted |
| `social_game` | correct_vote_ratio × (1 − target_rate / max) | Alignment × avoidance interaction |
| `negation_rate` | votesnegated / tribalsattended | Idol / advantage usage |
| `tribe_immunity_rate` | tribeimmunities / tribalsattended | Tribe challenge strength |
| `swap_resilience` | made_merge × (1 / (1 + timesswapped)) | Surviving swaps to reach the merge |

### Variables eliminated and why
| Eliminated | Reason |
|---|---|
| `pressure_index` | r = +0.99 with `target_rate` — mathematically almost identical |
| `threat_avoidance` | r = −0.82 with `target_rate` — just its inverse, no new information |
| `social_game` | r = +0.98 with `correct_vote_ratio` — dominated by one component |
| `negation_rate` | AUC = 0.556 — barely above random |
| `tribe_immunity_rate` | AUC = 0.621 — weak signal, captures team luck not individual strategy |
| `swap_resilience` | AUC = 0.698 — mostly captures whether someone made the merge (covered by `tribal_depth`) |
| `longevity` | **Data leakage** — 1 − (finalplacement / playersonseason) directly encodes the target |

In [ ]:
# ── Build all candidate variables ─────────────────────────────────────────────
df_work['is_winner'] = (df_work['finalplacement'] == 1).astype(int)

_cvr = np.where(df_work['votescast'] > 0, df_work['correctlyvoted'] / df_work['votescast'], np.nan)
_tgt = df_work['votesrecieved'] / df_work['tribalsattended']

df_work['target_rate']         = _tgt
df_work['correct_vote_ratio']  = _cvr
df_work['immunity_rate']       = df_work['individualimmunites'] / df_work['tribalsattended']
df_work['tribal_depth']        = df_work['tribalsattended'] / (df_work['playersonseason'] - 1)
df_work['pressure_index']      = (df_work['votesrecieved'] - df_work['votesnegated']) / df_work['tribalsattended']
df_work['threat_avoidance']    = 1 / (1 + _tgt)
df_work['social_game']         = _cvr * (1 - _tgt / df_work['votesrecieved'].max())
df_work['negation_rate']       = df_work['votesnegated'] / df_work['tribalsattended']
df_work['tribe_immunity_rate'] = df_work['tribeimmunities'] / df_work['tribalsattended']
df_work['swap_resilience']     = (df_work['mergetribe'] != 'No Merge').astype(int) * (1 / (1 + df_work['timesswapped']))

ALL_CANDIDATES = [
    'target_rate', 'correct_vote_ratio', 'immunity_rate', 'tribal_depth',
    'pressure_index', 'threat_avoidance', 'social_game',
    'negation_rate', 'tribe_immunity_rate', 'swap_resilience',
]

df_all = df_work.dropna(subset=ALL_CANDIDATES + ['is_winner']).copy()
y_all  = df_all['is_winner']
skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Candidate test dataset: {len(df_all)} rows, {y_all.sum()} winners")

In [ ]:
# ── AUC screening — all candidates ───────────────────────────────────────────
print(f"{'Variable':<24} {'AUC':>6}  {'+-':>5}  {'Win mean':>9}  {'Non-Win':>9}  Status")
print('-' * 78)

candidate_aucs = {}
ELIMINATED = {
    'pressure_index':      'REDUNDANT  (r=+0.99 with target_rate)',
    'threat_avoidance':    'REDUNDANT  (r=-0.82 with target_rate — inverse)',
    'social_game':         'REDUNDANT  (r=+0.98 with correct_vote_ratio)',
    'negation_rate':       'WEAK       (AUC<0.65)',
    'tribe_immunity_rate': 'WEAK       (AUC<0.65, captures team luck)',
    'swap_resilience':     'WEAK       (AUC=0.698, covered by tribal_depth)',
}

for var in ALL_CANDIDATES:
    sc   = StandardScaler()
    X    = sc.fit_transform(df_all[[var]])
    lr   = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    aucs = cross_val_score(lr, X, y_all, cv=skf, scoring='roc_auc')
    wm   = df_all[df_all['is_winner']==1][var].mean()
    nm   = df_all[df_all['is_winner']==0][var].mean()
    candidate_aucs[var] = aucs.mean()
    status = ELIMINATED.get(var, 'SELECTED')
    flag   = '  <--' if status == 'SELECTED' else ''
    print(f"{var:<24} {aucs.mean():.3f}  +-{aucs.std():.3f}  {wm:>9.3f}  {nm:>9.3f}  {status}{flag}")

In [ ]:
# ── Correlation matrix — confirm no redundancy in final 4 ────────────────────
FINAL_VARS = ['target_rate', 'correct_vote_ratio', 'tribal_depth', 'immunity_rate']
print("=== Correlation matrix — Final 4 ===")
print("(Values > 0.90 would indicate redundancy — none found)")
print()
print(df_all[FINAL_VARS].corr().round(3).to_string())

---
## 3. Final 4 Variables — Construction & Validation

| # | Variable | Formula | Range | What it captures | Direction |
|---|---|---|---|---|---|
| 1 | **target_rate** | votesrecieved / tribalsattended | [0, ∞) | How threatening others perceive you | **↓ lower = better** |
| 2 | **correct_vote_ratio** | correctlyvoted / votescast | [0, 1] | Alignment with the majority vote | **↑ higher = better** |
| 3 | **tribal_depth** | tribalsattended / (playersonseason − 1) | [0, 1] | How deep into the game you survived | **↑ higher = better** |
| 4 | **immunity_rate** | individualimmunites / tribalsattended | [0, 1] | Individual challenge dominance | **↑ higher = better** |

**Why these four specifically:**
- `target_rate` — strongest single predictor (AUC 0.885). Being perceived as a threat is the #1 predictor of *not* winning.
- `correct_vote_ratio` — strong (AUC 0.841) and captures something different: social/political read of the game.
- `tribal_depth` — strong (AUC 0.863) and uniquely captures *endurance/longevity* — how long you survived relative to season size, independent of how you voted.
- `immunity_rate` — weaker alone (AUC 0.757) but contributes unique information (max correlation with others is only 0.33), representing a distinct pathway to safety.

**No leakage:** `longevity` (1 − finalplacement/playersonseason) was excluded because it directly encodes the outcome. `tribal_depth` uses *tribals attended*, not final placement — a player can attend many tribals and still lose.

In [ ]:
# ── Build final clean dataset ─────────────────────────────────────────────────
df_clean = df_work.dropna(subset=FINAL_VARS + ['is_winner']).copy()

print("=== Range validation ===")
checks = {
    'target_rate >= 0'              : (df_clean['target_rate'] >= 0).all(),
    'correct_vote_ratio in [0,1]'   : df_clean['correct_vote_ratio'].between(0, 1).all(),
    'tribal_depth in [0,1]'         : df_clean['tribal_depth'].between(0, 1).all(),
    'immunity_rate in [0,1]'        : df_clean['immunity_rate'].between(0, 1).all(),
    'is_winner in {0,1}'            : df_clean['is_winner'].isin([0,1]).all(),
    'one winner per season'         : (df_clean.groupby('seasonplayed')['is_winner'].sum() == 1).all(),
    'no nulls in key vars'          : df_clean[FINAL_VARS].isnull().sum().sum() == 0,
}
for check, ok in checks.items():
    print(f"  {'OK  ' if ok else 'FAIL'}  {check}")

print(f"\nFinal dataset: {len(df_clean)} rows | {df_clean['is_winner'].sum()} winners")
print()
print("=== Winner vs Non-Winner means ===")
comp = df_clean.groupby('is_winner')[FINAL_VARS].mean().round(3)
comp.index = ['Non-Winner', 'Winner']
print(comp)

print()
print("=== Pearson r with finalplacement ===")
for var in FINAL_VARS:
    r, p = pearsonr(df_clean[var], df_clean['finalplacement'])
    sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
    print(f"  {var:<26}  r = {r:+.3f}  p = {p:.2e}  {sig}")

---
## 4. Era Segmentation

In [ ]:
season_to_num = {
    'Borneo': 1, 'Australian Outback': 2, 'Africa': 3, 'Marquesas': 4,
    'Thailand': 5, 'The Amazon': 6, 'Pearl Islands': 7, 'All-Stars': 8,
    'Vanuatu': 9, 'Palau': 10, 'Guatemala': 11, 'Panama': 12,
    'Cook Islands': 13, 'Fiji': 14, 'China': 15, 'Micronesia': 16,
    'Gabon': 17, 'Tocantins': 18, 'Samoa': 19, 'Heroes vs. Villains': 20,
    'Nicaragua': 21, 'Redemption Island': 22, 'South Pacific': 23,
    'Philippines': 24, 'One World': 25, 'Caramoan': 26, 'Blood vs. Water': 27,
    'Cagayan': 28, 'San Juan del Sur': 29, 'Worlds Apart': 30,
    'Cambodia': 31, 'Kaôh Rōng': 32, 'Millenials vs. Gen X': 33,
    'Game Changers': 34, 'Heroes vs. Healers vs. Hustlers': 35,
    'Ghost Island': 36, 'David vs. Goliath': 37, 'Edge of Extinction': 38,
    'Island of the Idols': 39, 'Winners at War': 40,
    'Survivor 41': 41, 'Survivor 42': 42, 'Survivor 43': 43,
    'Survivor 44': 44, 'Survivor 45': 45, 'Survivor 46': 46,
    'Survivor 47': 47, 'Survivor 48': 48,
}

def assign_era(n):
    if pd.isnull(n):  return 'Unknown'
    if n <= 10:       return 'Old School (1-10)'
    if n <= 20:       return 'Middle School (11-20)'
    if n <= 27:       return 'Dark Era (21-27)'
    if n <= 40:       return 'Advantages Era (28-40)'
    return 'New Era (41+)'

df_clean['season_num'] = df_clean['seasonplayed'].map(season_to_num)
df_clean['era']        = df_clean['season_num'].apply(assign_era)

ERA_ORDER = [
    'Old School (1-10)', 'Middle School (11-20)', 'Dark Era (21-27)',
    'Advantages Era (28-40)', 'New Era (41+)'
]

df_adv = df_clean[df_clean['era'] == 'Advantages Era (28-40)'].copy()

print(df_clean.groupby('era').agg(
    players=('playername','count'),
    seasons=('seasonplayed','nunique'),
    winners=('is_winner','sum')
).reindex(ERA_ORDER))

---
## 5. Logistic Regression

> All features are **z-scored** (StandardScaler) before fitting so coefficients are on the same scale and directly comparable as importance weights.

In [ ]:
def run_logistic(X_df, y, label=''):
    scaler = StandardScaler()
    X_s    = scaler.fit_transform(X_df)
    lr     = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs   = cross_val_score(lr, X_s, y, cv=skf, scoring='roc_auc')
    lr.fit(X_s, y)
    return {
        'label'   : label,
        'auc_mean': aucs.mean(),
        'auc_std' : aucs.std(),
        'coefs'   : dict(zip(X_df.columns, lr.coef_[0])),
    }

In [ ]:
# ── 5a. Univariate — full dataset ─────────────────────────────────────────────
y_full = df_clean['is_winner']
uni_results = {}

print("=== Univariate AUC — Full Dataset ===")
print(f"{'Variable':<26} {'AUC':>6}  {'+-':>5}  {'Coef':>8}  Direction")
print('-' * 68)
for var in FINAL_VARS:
    res  = run_logistic(df_clean[[var]], y_full, label=var)
    uni_results[var] = res
    coef = res['coefs'][var]
    dirn = 'more -> win' if coef > 0 else 'less -> win'
    print(f"{var:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  {coef:>+8.3f}  {dirn}")

best = max(uni_results, key=lambda v: uni_results[v]['auc_mean'])
print(f"\nStrongest single predictor: {best}  (AUC {uni_results[best]['auc_mean']:.3f})")

In [ ]:
# ── 5b. Combined model — full dataset ─────────────────────────────────────────
combined_full = run_logistic(df_clean[FINAL_VARS], y_full, label='Combined')

print("=== Combined Model — Full Dataset ===")
print(f"ROC-AUC: {combined_full['auc_mean']:.3f} +- {combined_full['auc_std']:.3f}")
print()
print("Standardised coefficients (ranked by |magnitude| = importance):")
for feat, coef in sorted(combined_full['coefs'].items(), key=lambda x: abs(x[1]), reverse=True):
    bar  = 'X' * int(abs(coef) * 6)
    sign = '+' if coef > 0 else '-'
    print(f"  {feat:<26} {sign}{abs(coef):.3f}  {bar}")

In [ ]:
# ── 5c. Univariate + combined — Advantages Era ───────────────────────────────
y_adv     = df_adv['is_winner']
uni_adv   = {}

print("=== Univariate AUC — Advantages Era ===")
print(f"{'Variable':<26} {'AUC':>6}  {'+-':>5}  {'Coef':>8}")
print('-' * 52)
for var in FINAL_VARS:
    res = run_logistic(df_adv[[var]], y_adv, label=var)
    uni_adv[var] = res
    print(f"{var:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  {res['coefs'][var]:>+8.3f}")

print()
combined_adv = run_logistic(df_adv[FINAL_VARS], y_adv)
print(f"=== Combined Model — Advantages Era ===")
print(f"ROC-AUC: {combined_adv['auc_mean']:.3f} +- {combined_adv['auc_std']:.3f}")
print()
for feat, coef in sorted(combined_adv['coefs'].items(), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:<26} {coef:+.3f}")

In [ ]:
# ── 5d. Era-by-era combined models ────────────────────────────────────────────
era_results = {}

print(f"{'Era':<26} {'AUC':>6}  {'+-':>5}  | {'tgt':>6}  {'cvr':>6}  {'depth':>6}  {'imm':>6}")
print('-' * 74)
for era in ERA_ORDER:
    sub = df_clean[df_clean['era'] == era]
    if sub['is_winner'].sum() < 3:
        print(f"{era:<26}  -- insufficient winners")
        continue
    res = run_logistic(sub[FINAL_VARS], sub['is_winner'], label=era)
    era_results[era] = res
    c = res['coefs']
    print(f"{era:<26} {res['auc_mean']:.3f}  +-{res['auc_std']:.3f}  | "
          f"{c['target_rate']:+.2f}  "
          f"{c['correct_vote_ratio']:+.2f}  "
          f"{c['tribal_depth']:+.2f}  "
          f"{c['immunity_rate']:+.2f}")

---
## 6. Visualizations

In [ ]:
# ── Plot 1: Full candidate AUC comparison (all 10, showing selection rationale) 
all_aucs    = {v: candidate_aucs[v] for v in ALL_CANDIDATES}
vars_sorted = sorted(all_aucs.items(), key=lambda x: x[1], reverse=True)
labels_all  = [v for v, _ in vars_sorted]
aucs_all    = [a for _, a in vars_sorted]
colors_all  = [BLUE if v in FINAL_VARS else '#CFD8DC' for v in labels_all]
edge_all    = [DBLUE if v in FINAL_VARS else GREY for v in labels_all]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels_all, aucs_all, color=colors_all,
               edgecolor=edge_all, linewidth=1.2, height=0.62)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=1, label='Random baseline')
ax.axvline(0.75, color=GREY, linestyle=':', linewidth=0.8, label='Weak threshold (0.75)')
for bar, auc, lbl in zip(bars, aucs_all, labels_all):
    col = 'white' if lbl in FINAL_VARS else 'black'
    ax.text(auc - 0.005, bar.get_y() + bar.get_height()/2,
            f'{auc:.3f}', va='center', ha='right', fontsize=9,
            color='white' if auc > 0.76 else 'black', fontweight='bold')

# Legend patches
from matplotlib.patches import Patch
legend_els = [
    Patch(facecolor=BLUE,    edgecolor=DBLUE, label='Selected (Final 4)'),
    Patch(facecolor='#CFD8DC', edgecolor=GREY, label='Eliminated'),
]
ax.legend(handles=legend_els + [plt.Line2D([0],[0], color='grey', linestyle='--', label='Random baseline')],
          fontsize=9, loc='lower right')
ax.set_xlabel('ROC-AUC (5-fold stratified CV)')
ax.set_title('All 10 candidate variables — AUC screening\nBlue = selected, Grey = eliminated', fontweight='bold')
ax.set_xlim(0.38, 1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Univariate AUC — Final 4 with winner/non-winner mean gap ──────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: AUC bars
ax = axes[0]
v_sorted = sorted(FINAL_VARS, key=lambda v: uni_results[v]['auc_mean'], reverse=True)
aucs_f   = [uni_results[v]['auc_mean'] for v in v_sorted]
stds_f   = [uni_results[v]['auc_std']  for v in v_sorted]
bars = ax.barh(v_sorted, aucs_f, xerr=stds_f, color=[BLUE, LBLUE, LBLUE, LBLUE],
               capsize=4, edgecolor='white', height=0.5)
ax.axvline(0.5, color='grey', linestyle='--', linewidth=1)
for bar, auc in zip(bars, aucs_f):
    ax.text(auc + 0.008, bar.get_y() + bar.get_height()/2,
            f'{auc:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0.4, 1.05)
ax.set_xlabel('ROC-AUC')
ax.set_title('Univariate predictive power\n(which variable matters most alone?)', fontweight='bold')

# Right: winner vs non-winner mean comparison
ax2 = axes[1]
x   = np.arange(len(FINAL_VARS))
w_means = df_clean[df_clean['is_winner']==1][FINAL_VARS].mean()
l_means = df_clean[df_clean['is_winner']==0][FINAL_VARS].mean()
# Normalise to [0,1] for visual comparison
all_max = df_clean[FINAL_VARS].max()
w_norm  = w_means / all_max
l_norm  = l_means / all_max
ax2.bar(x - 0.2, w_norm, 0.36, label='Winner mean',     color=BLUE,  alpha=0.85)
ax2.bar(x + 0.2, l_norm, 0.36, label='Non-Winner mean', color=RED,   alpha=0.75)
ax2.set_xticks(x)
ax2.set_xticklabels(['target_rate\n(inverted)', 'correct_vote\nratio', 'tribal\ndepth', 'immunity\nrate'], fontsize=9)
ax2.set_ylabel('Normalised mean (/ max)')
ax2.set_title('Winner vs Non-Winner means\n(normalised for comparison)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.text(0.02, 0.95, 'Note: target_rate inverted\n(lower = better for winners)',
         transform=ax2.transAxes, fontsize=7.5, va='top', color=GREY,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Distributions for each variable — Advantages Era ─────────────────
win_adv  = df_adv[df_adv['is_winner']==1]
lose_adv = df_adv[df_adv['is_winner']==0]

VAR_LABELS = {
    'target_rate'       : 'Target Rate   (lower = better)',
    'correct_vote_ratio': 'Correct Vote Ratio   (higher = better)',
    'tribal_depth'      : 'Tribal Depth   (higher = better)',
    'immunity_rate'     : 'Immunity Rate   (higher = better)',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, var in zip(axes.flatten(), FINAL_VARS):
    ax.hist(lose_adv[var], bins=20, alpha=0.50, color=RED,  label='Non-Winners', density=True)
    ax.hist(win_adv[var],  bins=20, alpha=0.80, color=BLUE, label='Winners',     density=True)
    wm = win_adv[var].mean()
    lm = lose_adv[var].mean()
    ax.axvline(wm, color=DBLUE, linestyle='--', linewidth=2, label=f'Winner mean: {wm:.3f}')
    ax.axvline(lm, color=DRED,  linestyle='--', linewidth=2, label=f'Non-winner mean: {lm:.3f}')
    ax.set_title(VAR_LABELS[var], fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle('Advantages Era — Distributions: Winners vs Non-Winners',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 4: AUC trend across eras ─────────────────────────────────────────────
era_labels = [e for e in ERA_ORDER if e in era_results]
era_aucs   = [era_results[e]['auc_mean'] for e in era_labels]
era_stds   = [era_results[e]['auc_std']  for e in era_labels]

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(era_labels,
                [a - s for a, s in zip(era_aucs, era_stds)],
                [a + s for a, s in zip(era_aucs, era_stds)],
                alpha=0.15, color=BLUE)
ax.plot(era_labels, era_aucs, 'o-', color=BLUE, linewidth=2.5,
        markersize=9, markerfacecolor='white', markeredgewidth=2.5)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='Random baseline')
for x_pos, auc in zip(era_labels, era_aucs):
    ax.annotate(f'{auc:.3f}', (x_pos, auc), textcoords='offset points',
                xytext=(0, 13), ha='center', fontsize=9, fontweight='bold', color=DBLUE)
ax.set_ylim(0.4, 1.1)
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Combined model AUC across eras\nHow well do these 4 variables predict winners in each period?', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 5: Coefficient heatmap — which variable drives each era ───────────────
coef_df = pd.DataFrame(
    {era: era_results[era]['coefs'] for era in era_labels}
).T[FINAL_VARS]

short_var_names = ['target\nrate', 'correct_vote\nratio', 'tribal\ndepth', 'immunity\nrate']

fig, ax = plt.subplots(figsize=(11, 4))
vmax = max(abs(coef_df.values.min()), abs(coef_df.values.max()))
im   = ax.imshow(coef_df.values, cmap='RdBu', aspect='auto', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(len(FINAL_VARS)))
ax.set_xticklabels(short_var_names)
ax.set_yticks(range(len(era_labels)))
ax.set_yticklabels(era_labels)
for i in range(len(era_labels)):
    for j in range(len(FINAL_VARS)):
        val = coef_df.iloc[i, j]
        ax.text(j, i, f'{val:+.2f}', ha='center', va='center', fontsize=10,
                color='white' if abs(val) > vmax * 0.55 else 'black')
plt.colorbar(im, ax=ax, label='Standardised coefficient')
ax.set_title('Variable importance by era\n'
             'Blue = positive (more -> winning)   Red = negative (less -> winning)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 6: Era averages — all 4 variables, winners vs non-winners ─────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
x_era  = np.arange(len(ERA_ORDER))
xlbls  = ['Old\nSchool', 'Middle\nSchool', 'Dark\nEra', 'Advantages\nEra', 'New\nEra']

for ax, var in zip(axes.flatten(), FINAL_VARS):
    w_vals = df_clean[df_clean['is_winner']==1].groupby('era')[var].mean().reindex(ERA_ORDER)
    l_vals = df_clean[df_clean['is_winner']==0].groupby('era')[var].mean().reindex(ERA_ORDER)
    ax.bar(x_era - 0.22, w_vals, 0.4, label='Winners',     color=BLUE, alpha=0.85)
    ax.bar(x_era + 0.22, l_vals, 0.4, label='Non-Winners', color=RED,  alpha=0.72)
    ax.set_xticks(x_era)
    ax.set_xticklabels(xlbls, fontsize=9)
    ax.set_title(VAR_LABELS[var], fontweight='bold')
    ax.legend(fontsize=8)

fig.suptitle('Winner vs Non-Winner Averages Across All Eras',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 7: Strategy quadrant — target_rate vs correct_vote_ratio ─────────────
# The two strongest predictors together. Winners should cluster bottom-right:
# high alignment + low targeting.
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(df_adv[df_adv['is_winner']==0]['correct_vote_ratio'],
           df_adv[df_adv['is_winner']==0]['target_rate'],
           c=RED,  s=30,  alpha=0.55, label='Non-Winner', zorder=2)
ax.scatter(df_adv[df_adv['is_winner']==1]['correct_vote_ratio'],
           df_adv[df_adv['is_winner']==1]['target_rate'],
           c=BLUE, s=130, alpha=0.90, label='Winner', marker='*', zorder=3)

xm = df_adv['correct_vote_ratio'].median()
ym = df_adv['target_rate'].median()
ax.axvline(xm, color='grey', linestyle=':', linewidth=0.9)
ax.axhline(ym, color='grey', linestyle=':', linewidth=0.9)

fs = 8.5
ax.text(0.02, 0.98, 'Misaligned\n+ Targeted',  transform=ax.transAxes, va='top',    fontsize=fs, color=DRED)
ax.text(0.68, 0.98, 'Aligned\n+ Targeted',      transform=ax.transAxes, va='top',    fontsize=fs, color='#6A1B9A')
ax.text(0.02, 0.02, 'Misaligned\n+ Safe',        transform=ax.transAxes, va='bottom', fontsize=fs, color='#E65100')
ax.text(0.68, 0.02, 'Aligned + Safe\n<-- IDEAL', transform=ax.transAxes, va='bottom', fontsize=fs, color=DBLUE, fontweight='bold')

ax.set_xlabel('Correct Vote Ratio  ->  strategic alignment with majority', fontsize=11)
ax.set_ylabel('Target Rate  ->  threat perceived by opponents', fontsize=11)
ax.set_title('The winning profile: Advantages Era\n'
             'Winners cluster bottom-right — strategically aligned and avoided', fontweight='bold')
ax.legend(fontsize=11, markerscale=1.2)
plt.tight_layout()
plt.show()